# Policy Performance Sandbox

This notebook is a local shadow evaluator for `student_policies.ipynb`. It is meant to estimate possible performance, not official leaderboard performance.

## What This Does

- Loads your current `StudentPricingPolicy` and `StudentMatchingPolicy` directly from `student_policies.ipynb`.
- Replays the historical arrival stream in `data/training_data.csv`.
- Uses an empirical conversion proxy to estimate whether riders would accept your quoted price.
- Simulates matching and dispatch under a configurable batching horizon.
- Reports directional metrics such as quote level, conversion rate, shared-match rate, wait time, revenue, cost, and profit proxy.

## Important Caveat

This is only a **shadow test**. It is useful for comparing policy variants against each other, but it is not the course evaluator.

In [1]:
import json
import pickle
import copy
import math
import time
from collections import defaultdict

import numpy as np
import pandas as pd

from rider import rider
from utils import populate_shared_ride_lengths

POLICY_NOTEBOOKS = {
    "evelyn": "C:\\Users\\user\\Desktop\\C253\\Calyber\\calyber-gre\\student_policies_evelyn.ipynb",
    "grace": "C:\\Users\\user\\Desktop\\C253\\Calyber\\calyber-gre\\student_policies_Grace.ipynb",
    "ryan": "C:\\Users\\user\\Desktop\\C253\\Calyber\\calyber-gre\\student_policies_ryan.ipynb",
}

DATA_PATH = "data/training_data.csv"
COST_PER_MILE = 0.70
MAX_WAIT_SECONDS = 120
N_REPLICATIONS = 1
RANDOM_SEED = 42
SAMPLE_FRAC = 1.0
WAIT_GRID = (15, 60, 120)


In [11]:

# def load_policy_classes(policy_notebook=POLICY_NOTEBOOK):
#     with open(policy_notebook, "r", encoding="utf-8") as f:
#         nb = json.load(f)

#     namespace = {}
#     code_cells = [cell for cell in nb["cells"] if cell.get("cell_type") == "code"]

#     for cell in code_cells:
#         source = "".join(cell.get("source", []))
#         exec(source, namespace)

#     return namespace["StudentPricingPolicy"], namespace["StudentMatchingPolicy"], namespace


# StudentPricingPolicy, StudentMatchingPolicy, policy_namespace = load_policy_classes()
# print("Loaded pricing policy:", StudentPricingPolicy.get_name())
# print("Loaded matching policy:", StudentMatchingPolicy.get_name())


def load_policy_classes(policy_notebook):
    with open(policy_notebook, "r", encoding="utf-8") as f:
        nb = json.load(f)

    namespace = {}
    code_cells = [cell for cell in nb["cells"] if cell.get("cell_type") == "code"]

    for cell in code_cells:
        source = "".join(cell.get("source", []))
        exec(source, namespace)

    return namespace["StudentPricingPolicy"], namespace["StudentMatchingPolicy"], namespace


# Load all policies from each notebook
all_policies = {}
for name, path in POLICY_NOTEBOOKS.items():
    pricing, matching, ns = load_policy_classes(path)
    all_policies[name] = {
        "pricing": pricing,
        "matching": matching,
        "namespace": ns,
    }
    print(f"Loaded [{name}] pricing: {pricing.get_name()} | matching: {matching.get_name()}")


# Generate all 9 pairs (3x3 cartesian product: each pricing vs each matching)
policy_pairs = []
for pricing_name, pricing_data in all_policies.items():
    for matching_name, matching_data in all_policies.items():
        pair = {
            "label": f"{pricing_name}_pricing + {matching_name}_matching",
            "pricing_policy": pricing_data["pricing"],
            "matching_policy": matching_data["matching"],
        }
        policy_pairs.append(pair)

print(f"\nGenerated {len(policy_pairs)} policy pairs:")
for i, pair in enumerate(policy_pairs, 1):
    print(f"  {i}. {pair['label']}")

Loaded [evelyn] pricing: GRE | matching: GRE

=============== Pricing at State 0 (0 waiting requests) ===============
Pricing decision: 0.70000.
Execution time of the pricing function is 0.00000 seconds.

=============== Matching at State 0 (0 waiting requests) ===============
Matching decision: do not match.
Execution time of the matching function is 0.00000 seconds.

=============== Pricing at State 1 (8 waiting requests) ===============
Pricing decision: 0.64604.
Execution time of the pricing function is 0.00201 seconds.

=============== Matching at State 1 (8 waiting requests) ===============
Matching decision: match the incoming rider with a waiting request.
Execution time of the matching function is 0.00000 seconds.

=============== Pricing at State 2 (35 waiting requests) ===============
Pricing decision: 0.66078.
Execution time of the pricing function is 0.00100 seconds.

=============== Matching at State 2 (35 waiting requests) ===============
Matching decision: match the inco

In [12]:
df = pd.read_csv(DATA_PATH).sort_values(["arrival_week", "arrival_time", "rider_id"]).reset_index(drop=True)

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_SEED).sort_values(["arrival_week", "arrival_time", "rider_id"]).reset_index(drop=True)

PRICE_BINS = np.linspace(0.0, 1.0, 21)
LENGTH_BINS = [0.0, 1.5, 2.5, 4.0, 6.0, 9.0, float("inf")]
GLOBAL_CONVERSION = float(df["convert_or_not"].mean())

def price_bucket(price):
    clipped = float(np.clip(price, 0.0, 1.0))
    idx = int(np.digitize([clipped], PRICE_BINS[1:-1], right=False)[0])
    return idx

def length_bucket(length):
    idx = int(np.digitize([float(length)], LENGTH_BINS[1:-1], right=False)[0])
    return idx

def smoothed_rate(table, key, base_rate, strength):
    if key not in table:
        return base_rate
    count, mean = table[key]
    return (count * mean + strength * base_rate) / (count + strength)

def build_rate_table(group_cols, dataframe):
    grouped = dataframe.groupby(group_cols)["convert_or_not"].agg(["count", "mean"]).reset_index()
    table = {}
    for row in grouped.itertuples(index=False):
        key = tuple(getattr(row, col) for col in group_cols)
        table[key] = (int(row.count), float(row.mean))
    return table

df_proxy = df.copy()
df_proxy["price_bucket"] = df_proxy["quoted_price"].apply(price_bucket)
df_proxy["length_bucket"] = df_proxy["solo_length"].apply(length_bucket)
accepted_df = df_proxy[df_proxy["convert_or_not"] == 1].copy()

price_bucket_summary = df_proxy.groupby("price_bucket")["convert_or_not"].agg(["count", "mean"]).reset_index()
PRICE_BUCKET_CENTERS = np.array(
    [(PRICE_BINS[int(idx)] + PRICE_BINS[int(idx) + 1]) / 2.0 for idx in price_bucket_summary["price_bucket"]],
    dtype=float,
)
PRICE_BUCKET_RATES = price_bucket_summary["mean"].to_numpy(dtype=float).copy()
for idx in range(1, len(PRICE_BUCKET_RATES)):
    PRICE_BUCKET_RATES[idx] = min(PRICE_BUCKET_RATES[idx], PRICE_BUCKET_RATES[idx - 1])
GLOBAL_BUCKET_BASE_RATES = {
    int(bucket): max(0.01, float(rate))
    for bucket, rate in zip(price_bucket_summary["price_bucket"], PRICE_BUCKET_RATES)
}

LENGTH_PRICE_TABLE = build_rate_table(["length_bucket", "price_bucket"], df_proxy)
ROUTE_PRICE_TABLE = build_rate_table(["pickup_area", "dropoff_area", "price_bucket"], df_proxy)
ROUTE_LENGTH_PRICE_TABLE = build_rate_table(["pickup_area", "dropoff_area", "length_bucket", "price_bucket"], df_proxy)

ACCEPTED_ROUTE_PRICE_CAP = accepted_df.groupby(["pickup_area", "dropoff_area"])["quoted_price"].max().to_dict()
ACCEPTED_LENGTH_PRICE_CAP = accepted_df.groupby("length_bucket")["quoted_price"].max().to_dict()
GLOBAL_MAX_ACCEPTED_QUOTE = float(accepted_df["quoted_price"].max())


def continuous_price_probability(quoted_price):
    return float(
        np.interp(
            [float(quoted_price)],
            PRICE_BUCKET_CENTERS,
            PRICE_BUCKET_RATES,
            left=PRICE_BUCKET_RATES[0],
            right=PRICE_BUCKET_RATES[-1],
        )[0]
    )


def estimate_conversion_probability(row_like, quoted_price):
    p_bucket = price_bucket(quoted_price)
    l_bucket = length_bucket(row_like["solo_length"])
    route_key = (int(row_like["pickup_area"]), int(row_like["dropoff_area"]))

    prob = GLOBAL_BUCKET_BASE_RATES[p_bucket]
    prob = smoothed_rate(LENGTH_PRICE_TABLE, (l_bucket, p_bucket), prob, 18.0)
    prob = smoothed_rate(ROUTE_PRICE_TABLE, route_key + (p_bucket,), prob, 12.0)
    prob = smoothed_rate(ROUTE_LENGTH_PRICE_TABLE, route_key + (l_bucket, p_bucket), prob, 6.0)

    # Preserve within-bucket price sensitivity instead of treating the whole bucket as flat.
    prob *= continuous_price_probability(quoted_price) / max(GLOBAL_BUCKET_BASE_RATES[p_bucket], 1e-6)

    # Penalize quotes that exceed historically supported accepted prices for this route/length.
    route_cap = ACCEPTED_ROUTE_PRICE_CAP.get(route_key, GLOBAL_MAX_ACCEPTED_QUOTE)
    length_cap = ACCEPTED_LENGTH_PRICE_CAP.get(l_bucket, GLOBAL_MAX_ACCEPTED_QUOTE)
    support_cap = min(max(route_cap, length_cap), GLOBAL_MAX_ACCEPTED_QUOTE)

    if quoted_price > support_cap:
        prob *= math.exp(-18.0 * (float(quoted_price) - support_cap))
    if quoted_price > GLOBAL_MAX_ACCEPTED_QUOTE + 0.01:
        prob *= 0.25

    return float(np.clip(prob, 0.01, 0.98))


def row_to_rider(row_like):
    return rider(
        int(row_like["arrival_week"]),
        float(row_like["arrival_time"]),
        float(row_like["pickup_lat"]),
        float(row_like["pickup_lon"]),
        float(row_like["dropoff_lat"]),
        float(row_like["dropoff_lon"]),
        int(row_like["pickup_area"]),
        int(row_like["dropoff_area"]),
    )


def pair_cost_lengths(rider_i, rider_j):
    origin_i = (rider_i.pickup_lat, rider_i.pickup_lon)
    dest_i = (rider_i.dropoff_lat, rider_i.dropoff_lon)
    origin_j = (rider_j.pickup_lat, rider_j.pickup_lon)
    dest_j = (rider_j.dropoff_lat, rider_j.dropoff_lon)

    trip_length, shared_length, i_solo_length, j_solo_length, _ = populate_shared_ride_lengths(
        origin_i,
        dest_i,
        origin_j,
        dest_j,
    )

    return float(i_solo_length + shared_length / 2.0), float(j_solo_length + shared_length / 2.0), float(trip_length)

print(f"Loaded {len(df):,} riders for shadow evaluation.")
print(f"Global historical conversion rate: {GLOBAL_CONVERSION:.3f}")
print(f"Global max accepted quote: {GLOBAL_MAX_ACCEPTED_QUOTE:.3f}")

Loaded 11,788 riders for shadow evaluation.
Global historical conversion rate: 0.582
Global max accepted quote: 0.845


In [13]:
def initialize_metrics():
    return defaultdict(float)


def dispatch_solo(record, dispatch_time, metrics):
    wait_time = max(0.0, float(dispatch_time) - float(record["arrival_time"]))
    revenue = float(record["price"]) * float(record["solo_length"])
    cost = COST_PER_MILE * float(record["solo_length"])

    metrics["converted_riders"] += 1
    metrics["solo_dispatches"] += 1
    metrics["revenue"] += revenue
    metrics["cost"] += cost
    metrics["profit"] += revenue - cost
    metrics["total_wait_time"] += wait_time


def dispatch_shared(incoming_record, waiting_record, current_time, metrics):
    cost_incoming, cost_waiting, trip_length = pair_cost_lengths(incoming_record["rider"], waiting_record["rider"])

    revenue_incoming = float(incoming_record["price"]) * float(incoming_record["solo_length"])
    revenue_waiting = float(waiting_record["price"]) * float(waiting_record["solo_length"])

    wait_incoming = max(0.0, float(current_time) - float(incoming_record["arrival_time"]))
    wait_waiting = max(0.0, float(current_time) - float(waiting_record["arrival_time"]))

    metrics["converted_riders"] += 2
    metrics["shared_riders"] += 2
    metrics["matched_pairs"] += 1
    metrics["revenue"] += revenue_incoming + revenue_waiting
    metrics["cost"] += COST_PER_MILE * (cost_incoming + cost_waiting)
    metrics["profit"] += revenue_incoming + revenue_waiting - COST_PER_MILE * (cost_incoming + cost_waiting)
    metrics["total_wait_time"] += wait_incoming + wait_waiting
    metrics["total_trip_length"] += trip_length


def flush_expired(queue, current_time, max_wait_seconds, metrics):
    still_waiting = []
    for record in queue:
        if float(current_time) - float(record["arrival_time"]) >= max_wait_seconds:
            dispatch_solo(record, record["arrival_time"] + max_wait_seconds, metrics)
        else:
            still_waiting.append(record)
    return still_waiting


def finalize_queue(queue, max_wait_seconds, metrics):
    for record in queue:
        dispatch_solo(record, record["arrival_time"] + max_wait_seconds, metrics)


def summarize_metrics(metrics):
    summary = dict(metrics)
    requests = max(summary.get("requests", 0.0), 1.0)
    converted = max(summary.get("converted_riders", 0.0), 1.0)

    summary["avg_quote"] = summary.get("quoted_price_sum", 0.0) / requests
    summary["expected_conversion_rate"] = summary.get("convert_prob_sum", 0.0) / requests
    summary["simulated_conversion_rate"] = summary.get("converted_riders", 0.0) / requests
    summary["shared_rate_given_conversion"] = summary.get("shared_riders", 0.0) / converted
    summary["avg_wait_time"] = summary.get("total_wait_time", 0.0) / converted
    summary["profit_per_request"] = summary.get("profit", 0.0) / requests
    summary["profit_per_converted_rider"] = summary.get("profit", 0.0) / converted
    summary["revenue_per_request"] = summary.get("revenue", 0.0) / requests
    summary["cost_per_request"] = summary.get("cost", 0.0) / requests

    return summary


In [14]:
def historical_cost_length(row_like, rider_lookup):
    if pd.isna(row_like["matching_outcome"]):
        return float(row_like["solo_length"])

    matched_rider_id = int(row_like["matching_outcome"])
    if matched_rider_id not in rider_lookup:
        return float(row_like["solo_length"])

    matched_row = rider_lookup[matched_rider_id]

    origin_i = (float(row_like["pickup_lat"]), float(row_like["pickup_lon"]))
    dest_i = (float(row_like["dropoff_lat"]), float(row_like["dropoff_lon"]))
    origin_j = (float(matched_row["pickup_lat"]), float(matched_row["pickup_lon"]))
    dest_j = (float(matched_row["dropoff_lat"]), float(matched_row["dropoff_lon"]))

    trip_length, shared_length, i_solo_length, j_solo_length, _ = populate_shared_ride_lengths(origin_i, dest_i, origin_j, dest_j)
    return float(i_solo_length + shared_length / 2.0)


def historical_baseline_summary(dataframe):
    rider_lookup = dataframe.set_index("rider_id")[["pickup_lat", "pickup_lon", "dropoff_lat", "dropoff_lon"]].to_dict("index")

    converted = dataframe[dataframe["convert_or_not"] == 1].copy()
    converted["revenue"] = converted["quoted_price"] * converted["solo_length"]
    converted["cost_length"] = converted.apply(lambda row: historical_cost_length(row, rider_lookup), axis=1)
    converted["cost"] = COST_PER_MILE * converted["cost_length"]
    converted["profit"] = converted["revenue"] - converted["cost"]

    summary = {
        "requests": float(len(dataframe)),
        "avg_quote": float(dataframe["quoted_price"].mean()),
        "expected_conversion_rate": float(dataframe["convert_or_not"].mean()),
        "simulated_conversion_rate": float(dataframe["convert_or_not"].mean()),
        "shared_rate_given_conversion": float(converted["matching_outcome"].notna().mean()) if len(converted) else 0.0,
        "avg_wait_time": float(converted["waiting_time"].fillna(0.0).mean()) if len(converted) else 0.0,
        "revenue": float(converted["revenue"].sum()),
        "cost": float(converted["cost"].sum()),
        "profit": float(converted["profit"].sum()),
        "profit_per_request": float(converted["profit"].sum() / max(len(dataframe), 1)),
        "profit_per_converted_rider": float(converted["profit"].sum() / max(len(converted), 1)),
        "matched_pairs": float(converted["matching_outcome"].notna().sum() / 2.0),
    }

    return pd.Series(summary, name="historical_baseline")


baseline = historical_baseline_summary(df)
baseline



requests                        11788.000000
avg_quote                           0.558944
expected_conversion_rate            0.581948
simulated_conversion_rate           0.581948
shared_rate_given_conversion        0.798834
avg_wait_time                      26.811962
revenue                         12818.300696
cost                            14488.055691
profit                          -1669.754995
profit_per_request                 -0.141649
profit_per_converted_rider         -0.243405
matched_pairs                    2740.000000
Name: historical_baseline, dtype: float64

In [15]:
# def simulate_once(dataframe, max_wait_seconds=MAX_WAIT_SECONDS, seed=RANDOM_SEED):
#     rng = np.random.default_rng(seed)
#     pricing_policy = StudentPricingPolicy()
#     matching_policy = StudentMatchingPolicy()
#     metrics = initialize_metrics()

#     queue = []
#     current_week = None

#     for row in dataframe.itertuples(index=False):
#         row_dict = row._asdict()

#         if current_week is None:
#             current_week = int(row_dict["arrival_week"])
#         elif int(row_dict["arrival_week"]) != current_week:
#             finalize_queue(queue, max_wait_seconds, metrics)
#             queue = []
#             current_week = int(row_dict["arrival_week"])

#         current_time = float(row_dict["arrival_time"])
#         queue = flush_expired(queue, current_time, max_wait_seconds, metrics)

#         incoming_rider = row_to_rider(row_dict)
#         state = [record["rider"] for record in queue]
#         quoted_price = float(pricing_policy.pricing_function(state, incoming_rider))
#         convert_prob = estimate_conversion_probability(row_dict, quoted_price)

#         metrics["requests"] += 1
#         metrics["quoted_price_sum"] += quoted_price
#         metrics["convert_prob_sum"] += convert_prob

#         if rng.random() >= convert_prob:
#             continue

#         incoming_record = {
#             "rider": incoming_rider,
#             "arrival_week": int(row_dict["arrival_week"]),
#             "arrival_time": current_time,
#             "solo_length": float(row_dict["solo_length"]),
#             "price": quoted_price,
#             "rider_id": int(row_dict["rider_id"]),
#         }

#         matched_waiting_rider = matching_policy.matching_function(state, incoming_rider)
#         matched_index = None

#         if matched_waiting_rider is not None:
#             for idx, waiting_record in enumerate(queue):
#                 if waiting_record["rider"] is matched_waiting_rider:
#                     matched_index = idx
#                     break

#         if matched_index is not None:
#             waiting_record = queue.pop(matched_index)
#             dispatch_shared(incoming_record, waiting_record, current_time, metrics)
#         else:
#             queue.append(incoming_record)

#     finalize_queue(queue, max_wait_seconds, metrics)
#     return summarize_metrics(metrics)


# def evaluate_policy(wait_grid=WAIT_GRID, n_replications=N_REPLICATIONS, base_seed=RANDOM_SEED):
#     rows = []
#     for max_wait_seconds in wait_grid:
#         replications = []
#         for rep in range(n_replications):
#             replications.append(simulate_once(df, max_wait_seconds=max_wait_seconds, seed=base_seed + rep))

#         rep_df = pd.DataFrame(replications)
#         summary = rep_df.mean(numeric_only=True).to_dict()
#         summary["max_wait_seconds"] = max_wait_seconds
#         summary["n_replications"] = n_replications
#         rows.append(summary)

#     columns = [
#         "max_wait_seconds",
#         "n_replications",
#         "avg_quote",
#         "expected_conversion_rate",
#         "simulated_conversion_rate",
#         "shared_rate_given_conversion",
#         "avg_wait_time",
#         "revenue",
#         "cost",
#         "profit",
#         "profit_per_request",
#         "profit_per_converted_rider",
#         "matched_pairs",
#     ]

#     return pd.DataFrame(rows)[columns]


def simulate_once(dataframe, pricing_policy, matching_policy, max_wait_seconds=MAX_WAIT_SECONDS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    metrics = initialize_metrics()

    queue = []
    current_week = None

    for row in dataframe.itertuples(index=False):
        row_dict = row._asdict()

        if current_week is None:
            current_week = int(row_dict["arrival_week"])
        elif int(row_dict["arrival_week"]) != current_week:
            finalize_queue(queue, max_wait_seconds, metrics)
            queue = []
            current_week = int(row_dict["arrival_week"])

        current_time = float(row_dict["arrival_time"])
        queue = flush_expired(queue, current_time, max_wait_seconds, metrics)

        incoming_rider = row_to_rider(row_dict)
        state = [record["rider"] for record in queue]
        quoted_price = float(pricing_policy.pricing_function(state, incoming_rider))
        convert_prob = estimate_conversion_probability(row_dict, quoted_price)

        metrics["requests"] += 1
        metrics["quoted_price_sum"] += quoted_price
        metrics["convert_prob_sum"] += convert_prob

        if rng.random() >= convert_prob:
            continue

        incoming_record = {
            "rider": incoming_rider,
            "arrival_week": int(row_dict["arrival_week"]),
            "arrival_time": current_time,
            "solo_length": float(row_dict["solo_length"]),
            "price": quoted_price,
            "rider_id": int(row_dict["rider_id"]),
        }

        matched_waiting_rider = matching_policy.matching_function(state, incoming_rider)
        matched_index = None

        if matched_waiting_rider is not None:
            for idx, waiting_record in enumerate(queue):
                if waiting_record["rider"] is matched_waiting_rider:
                    matched_index = idx
                    break

        if matched_index is not None:
            waiting_record = queue.pop(matched_index)
            dispatch_shared(incoming_record, waiting_record, current_time, metrics)
        else:
            queue.append(incoming_record)

    finalize_queue(queue, max_wait_seconds, metrics)
    return summarize_metrics(metrics)


def evaluate_policy(pricing_policy, matching_policy, wait_grid=WAIT_GRID, n_replications=N_REPLICATIONS, base_seed=RANDOM_SEED):
    rows = []
    for max_wait_seconds in wait_grid:
        replications = []
        for rep in range(n_replications):
            replications.append(simulate_once(df, pricing_policy, matching_policy, max_wait_seconds=max_wait_seconds, seed=base_seed + rep))

        rep_df = pd.DataFrame(replications)
        summary = rep_df.mean(numeric_only=True).to_dict()
        summary["max_wait_seconds"] = max_wait_seconds
        summary["n_replications"] = n_replications
        rows.append(summary)

    columns = [
        "max_wait_seconds", "n_replications", "avg_quote",
        "expected_conversion_rate", "simulated_conversion_rate",
        "shared_rate_given_conversion", "avg_wait_time",
        "revenue", "cost", "profit",
        "profit_per_request", "profit_per_converted_rider", "matched_pairs",
    ]
    return pd.DataFrame(rows)[columns]


# # ── Pick one pair to evaluate ──────────────────────────────────────────────────
# PRICING_OWNER  = "evelyn"   # ← change these two to test a different pair
# MATCHING_OWNER = "grace"

# pricing_policy  = all_policies[PRICING_OWNER]["pricing"]()
# matching_policy = all_policies[MATCHING_OWNER]["matching"]()

# print(f"Evaluating: [{PRICING_OWNER}] pricing  +  [{MATCHING_OWNER}] matching")

# # ── Baseline ───────────────────────────────────────────────────────────────────
# baseline = historical_baseline_summary(df)

# # ── Shadow evaluation ──────────────────────────────────────────────────────────
# shadow_results = evaluate_policy(pricing_policy, matching_policy)

# # ── Delta vs baseline ──────────────────────────────────────────────────────────
# comparison = shadow_results.copy()
# for column in [
#     "avg_quote", "expected_conversion_rate", "simulated_conversion_rate",
#     "shared_rate_given_conversion", "avg_wait_time", "revenue", "cost",
#     "profit", "profit_per_request", "profit_per_converted_rider", "matched_pairs",
# ]:
#     comparison[f"delta_vs_baseline__{column}"] = comparison[column] - baseline[column]

# comparison

all_results = {}

for pair in policy_pairs:
    label          = pair["label"]
    pricing_policy  = pair["pricing_policy"]()
    matching_policy = pair["matching_policy"]()

    print(f"Evaluating: {label} ...")

    shadow_results = evaluate_policy(
        pricing_policy,
        matching_policy,
        wait_grid=WAIT_GRID,
        n_replications=N_REPLICATIONS,
        base_seed=RANDOM_SEED,
    )

    comparison = shadow_results.copy()
    for column in [
        "avg_quote", "expected_conversion_rate", "simulated_conversion_rate",
        "shared_rate_given_conversion", "avg_wait_time", "revenue", "cost",
        "profit", "profit_per_request", "profit_per_converted_rider", "matched_pairs",
    ]:
        comparison[f"delta_vs_baseline__{column}"] = comparison[column] - baseline[column]

    comparison["pair_label"] = label
    all_results[label] = comparison
    print(f"  ✓ Done: {label}")

# Combine all results into one DataFrame
combined = pd.concat(all_results.values(), ignore_index=True)
combined


Evaluating: evelyn_pricing + evelyn_matching ...
  ✓ Done: evelyn_pricing + evelyn_matching
Evaluating: evelyn_pricing + grace_matching ...
  ✓ Done: evelyn_pricing + grace_matching
Evaluating: evelyn_pricing + ryan_matching ...
  ✓ Done: evelyn_pricing + ryan_matching
Evaluating: grace_pricing + evelyn_matching ...
  ✓ Done: grace_pricing + evelyn_matching
Evaluating: grace_pricing + grace_matching ...
  ✓ Done: grace_pricing + grace_matching
Evaluating: grace_pricing + ryan_matching ...
  ✓ Done: grace_pricing + ryan_matching
Evaluating: ryan_pricing + evelyn_matching ...
  ✓ Done: ryan_pricing + evelyn_matching
Evaluating: ryan_pricing + grace_matching ...
  ✓ Done: ryan_pricing + grace_matching
Evaluating: ryan_pricing + ryan_matching ...
  ✓ Done: ryan_pricing + ryan_matching


,max_wait_seconds,n_replications,avg_quote,expected_conversion_rate,simulated_conversion_rate,shared_rate_given_conversion,avg_wait_time,revenue,cost,profit,...,delta_vs_baseline__simulated_conversion_rate,delta_vs_baseline__shared_rate_given_conversion,delta_vs_baseline__avg_wait_time,delta_vs_baseline__revenue,delta_vs_baseline__cost,delta_vs_baseline__profit,delta_vs_baseline__profit_per_request,delta_vs_baseline__profit_per_converted_rider,delta_vs_baseline__matched_pairs,pair_label
0,15,1,0.626900,0.474174,0.473448,0.972227,2.294571,11433.946211,12437.540284,-1003.594074,...,-0.108500,0.173393,-24.517391,-1384.354486,-2050.515407,666.160921,0.056512,0.063581,-27.0,evelyn_pricing + evelyn_matching
1,60,1,0.628146,0.472434,0.472769,0.999103,2.390095,11448.112182,12417.690802,-969.578620,...,-0.109179,0.200269,-24.421867,-1370.188514,-2070.364889,700.176375,0.059397,0.069427,44.0,evelyn_pricing + evelyn_matching
2,120,1,0.628146,0.472434,0.472769,0.999103,2.443926,11448.112182,12417.690802,-969.578620,...,-0.109179,0.200269,-24.368036,-1370.188514,-2070.364889,700.176375,0.059397,0.069427,44.0,evelyn_pricing + evelyn_matching
3,15,1,0.655739,0.422055,0.421021,0.398146,10.337094,10442.365146,10125.829399,316.535747,...,-0.160926,-0.400688,-16.474867,-2375.935550,-4362.226292,1986.290742,0.168501,0.307184,-1752.0,evelyn_pricing + grace_matching
4,60,1,0.657829,0.415901,0.415422,0.680417,26.237288,10347.509998,9440.112943,907.397055,...,-0.166525,-0.118417,-0.574674,-2470.790698,-5047.942748,2577.152050,0.218625,0.428701,-1074.0,evelyn_pricing + grace_matching
5,120,1,0.690889,0.355470,0.357482,0.766493,41.366160,9004.172268,7936.089928,1068.082340,...,-0.224466,-0.032341,14.554199,-3814.128429,-6551.965763,2737.837334,0.232256,0.496865,-1125.0,evelyn_pricing + grace_matching
6,15,1,0.655820,0.421719,0.420343,0.391927,10.409082,10438.657079,10132.492677,306.164402,...,-0.161605,-0.406906,-16.402880,-2379.643618,-4355.563014,1975.919396,0.167621,0.305193,-1769.0,evelyn_pricing + ryan_matching
7,60,1,0.658113,0.415391,0.417289,0.671275,26.886359,10358.015268,9470.781896,887.233371,...,-0.164659,-0.127559,0.074397,-2460.285429,-5017.273795,2556.988366,0.216915,0.423773,-1089.0,evelyn_pricing + ryan_matching
8,120,1,0.694376,0.348089,0.350696,0.748428,43.627479,8839.724574,7768.831468,1070.893107,...,-0.231252,-0.050406,16.815518,-3978.576122,-6719.224223,2740.648101,0.232495,0.502450,-1193.0,evelyn_pricing + ryan_matching
9,15,1,0.692838,0.347748,0.349678,0.964095,2.542213,9610.497392,9357.921514,252.575878,...,-0.232270,0.165261,-24.269749,-3207.803304,-5130.134177,1922.330873,0.163075,0.304680,-753.0,grace_pricing + evelyn_matching


In [16]:
combined.to_csv("policy_pair_evaluation_results_v2.csv", index=False)

In [17]:
shadow_results = evaluate_policy(
    pricing_policy,
    matching_policy,
    wait_grid=WAIT_GRID,
    n_replications=N_REPLICATIONS,
    base_seed=RANDOM_SEED,
)
shadow_results

,max_wait_seconds,n_replications,avg_quote,expected_conversion_rate,simulated_conversion_rate,shared_rate_given_conversion,avg_wait_time,revenue,cost,profit,profit_per_request,profit_per_converted_rider,matched_pairs
0,15,1,0.714791,0.310392,0.313794,0.283320,11.744526,8789.027968,7956.472211,832.555757,0.070627,0.225076,524.0
1,60,1,0.707136,0.325449,0.327961,0.566994,32.920590,9241.947200,7859.898191,1382.049008,0.117242,0.357488,1096.0
2,120,1,0.705744,0.327859,0.332117,0.685057,51.504981,9362.642731,7766.064786,1596.577945,0.135441,0.407810,1341.0


In [18]:
comparison = shadow_results.copy()
for column in [
    "avg_quote",
    "expected_conversion_rate",
    "simulated_conversion_rate",
    "shared_rate_given_conversion",
    "avg_wait_time",
    "revenue",
    "cost",
    "profit",
    "profit_per_request",
    "profit_per_converted_rider",
    "matched_pairs",
]:
    comparison[f"delta_vs_baseline__{column}"] = comparison[column] - baseline[column]

comparison


,max_wait_seconds,n_replications,avg_quote,expected_conversion_rate,simulated_conversion_rate,shared_rate_given_conversion,avg_wait_time,revenue,cost,profit,...,delta_vs_baseline__expected_conversion_rate,delta_vs_baseline__simulated_conversion_rate,delta_vs_baseline__shared_rate_given_conversion,delta_vs_baseline__avg_wait_time,delta_vs_baseline__revenue,delta_vs_baseline__cost,delta_vs_baseline__profit,delta_vs_baseline__profit_per_request,delta_vs_baseline__profit_per_converted_rider,delta_vs_baseline__matched_pairs
0,15,1,0.714791,0.310392,0.313794,0.283320,11.744526,8789.027968,7956.472211,832.555757,...,-0.271556,-0.268154,-0.515514,-15.067436,-4029.272728,-6531.583480,2502.310752,0.212276,0.468480,-2216.0
1,60,1,0.707136,0.325449,0.327961,0.566994,32.920590,9241.947200,7859.898191,1382.049008,...,-0.256499,-0.253987,-0.231840,6.108628,-3576.353497,-6628.157500,3051.804003,0.258891,0.600893,-1644.0
2,120,1,0.705744,0.327859,0.332117,0.685057,51.504981,9362.642731,7766.064786,1596.577945,...,-0.254089,-0.249830,-0.113776,24.693019,-3455.657965,-6721.990905,3266.332940,0.277090,0.651215,-1399.0


## Decide the best pair

In [19]:
# Find the best pair by average profit across all wait grid settings
best_by_profit = (
    combined.groupby("pair_label")["profit"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"profit": "avg_profit"})
)

print("Policy pairs ranked by average profit:\n")
print(best_by_profit.to_string(index=False))

best_label = best_by_profit.iloc[0]["pair_label"]
best_profit = best_by_profit.iloc[0]["avg_profit"]
print(f"\n🏆 Best pair: {best_label}")
print(f"   Average profit: {best_profit:,.4f}")

Policy pairs ranked by average profit:

                      pair_label  avg_profit
    ryan_pricing + ryan_matching 1270.394237
   ryan_pricing + grace_matching 1256.350948
  grace_pricing + grace_matching 1204.586610
   grace_pricing + ryan_matching 1196.990834
 evelyn_pricing + grace_matching  764.005047
  evelyn_pricing + ryan_matching  754.763626
  ryan_pricing + evelyn_matching  523.119784
 grace_pricing + evelyn_matching  249.229603
evelyn_pricing + evelyn_matching -980.917105

🏆 Best pair: ryan_pricing + ryan_matching
   Average profit: 1,270.3942
